In [168]:
import pandas as pd 
import numpy as np 
import os
import re
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, brier_score_loss
from xgboost import XGBClassifier

In [169]:
os.getcwd()
data = '/kaggle/input/competitions/march-machine-learning-mania-2026'
os.listdir(data)

['Cities.csv',
 'Conferences.csv',
 'MConferenceTourneyGames.csv',
 'MGameCities.csv',
 'MMasseyOrdinals.csv',
 'MNCAATourneyCompactResults.csv',
 'MNCAATourneyDetailedResults.csv',
 'MNCAATourneySeedRoundSlots.csv',
 'MNCAATourneySeeds.csv',
 'MNCAATourneySlots.csv',
 'MRegularSeasonCompactResults.csv',
 'MRegularSeasonDetailedResults.csv',
 'MSeasons.csv',
 'MSecondaryTourneyCompactResults.csv',
 'MSecondaryTourneyTeams.csv',
 'MTeamCoaches.csv',
 'MTeamConferences.csv',
 'MTeams.csv',
 'MTeamSpellings.csv',
 'SampleSubmissionStage1.csv',
 'SampleSubmissionStage2.csv',
 'WConferenceTourneyGames.csv',
 'WGameCities.csv',
 'WNCAATourneyCompactResults.csv',
 'WNCAATourneyDetailedResults.csv',
 'WNCAATourneySeeds.csv',
 'WNCAATourneySlots.csv',
 'WRegularSeasonCompactResults.csv',
 'WRegularSeasonDetailedResults.csv',
 'WSeasons.csv',
 'WSecondaryTourneyCompactResults.csv',
 'WSecondaryTourneyTeams.csv',
 'WTeamConferences.csv',
 'WTeams.csv',
 'WTeamSpellings.csv']

In [170]:
df_seeds = pd.concat(
    [pd.read_csv(data + "MNCAATourneySeeds.csv"),
    pd.read_csv(data + "WNCAATourneySeeds.csv")],
    axis=0
)
rank_avg = latest_rank.groupby(
    ["Season","TeamID"]
)["OrdinalRank"].mean().reset_index()

rank_avg = rank_avg.rename(columns={"OrdinalRank":"AvgRank"})
df_seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
1739,2025,Z12,3193
1740,2025,Z13,3251
1741,2025,Z14,3195
1742,2025,Z15,3117


In [171]:
df_tourney_results = pd.concat(
   [pd.read_csv(data + "MNCAATourneyCompactResults.csv"),
    pd.read_csv(data + "WNCAATourneyCompactResults.csv")],
    axis=0
)

df_tourney_results = df_tourney_results.drop(['NumOT', 'WLoc'], axis=1)
df_tourney_results



,Season,DayNum,WTeamID,WScore,LTeamID,LScore
0,1985,136,1116,63,1234,54
1,1985,136,1120,59,1345,58
2,1985,136,1207,68,1250,43
3,1985,136,1229,58,1425,55
4,1985,136,1242,49,1325,38
...,...,...,...,...,...,...
1712,2025,147,3163,78,3425,64
1713,2025,147,3400,58,3395,47
1714,2025,151,3163,85,3417,51
1715,2025,151,3376,74,3400,57


In [172]:
df_season_results = pd.concat(
   [pd.read_csv(data + "MRegularSeasonDetailedResults.csv"),
    pd.read_csv(data + "WRegularSeasonDetailedResults.csv")],
    axis=0
)
df_season_results

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86768,2026,118,3435,87,3397,77,A,0,35,66,...,21,9,10,9,20,9,7,2,13,14
86769,2026,118,3439,83,3438,82,A,0,27,65,...,15,18,25,9,22,15,4,3,14,19
86770,2026,118,3449,70,3332,69,A,0,27,59,...,16,13,16,9,20,13,6,2,14,16
86771,2026,118,3452,118,3153,60,H,0,42,72,...,11,14,24,6,19,9,2,1,17,25


In [173]:
def seed_to_num(seed):
    if pd.isna(seed):
        return np.nan
    m = re.search(r'(\d{1,2})', str(seed))
    return int(m.group(1)) if m else np.nan

df_seeds["SeedNum"] = df_seeds["Seed"].apply(seed_to_num)
df_seeds = df_seeds[["Season","TeamID","SeedNum"]]

In [174]:

w = df_season_results[["Season","WTeamID","WScore","LScore", "WFGA", "WOR", "WTO", "WFTA", "LFGA", "LOR", "LTO", "LFTA"]].copy()
w.columns = ["Season","TeamID","PF","PA", "FGA", "OR", "TO", "FTA", "OFGA", "OOR", "OTO", "OFTA"]
w["Win"] = 1


l = df_season_results[["Season","LTeamID","LScore","WScore", "LFGA","LOR", "LTO", "LFTA", "WFGA", "WOR", "WTO", "WFTA" ]].copy()
l.columns = ["Season","TeamID","PF","PA", "FGA", "OR", "TO", "FTA", "OFGA", "OOR", "OTO", "OFTA" ]
l["Win"] = 0


games = pd.concat([w,l], ignore_index=True)

team_season = games.groupby(["Season","TeamID"]).agg(
    Games=("Win","count"),
    Wins=("Win","sum"),
    PF=("PF","sum"),
    PA=("PA","sum"),
    FGA=("FGA","sum"),
    OR=("OR","sum"),
    TO=("TO","sum"),
    FTA=("FTA","sum"),
    OFGA=("OFGA","sum"),
    OOR=("OOR","sum"),
    OTO=("OTO","sum"),
    OFTA=("OFTA","sum")
).reset_index()

team_season["Losses"] = team_season["Games"] - team_season["Wins"]
team_season["WinPct"] = team_season["Wins"] / team_season["Games"]
team_season["AvgPF"] = team_season["PF"] / team_season["Games"]
team_season["AvgPA"] = team_season["PA"] / team_season["Games"]
team_season["AvgMargin"] = team_season["AvgPF"] - team_season["AvgPA"]
team_season["OffRating"] = team_season["PF"] /(100* (team_season["FGA"] + team_season["OR"] - team_season["TO"] + team_season["FTA"] * 0.475))
#National Average Offensive Efficiency 108.35, and the defense is 108.57
team_season["AdjOffRating"] = (team_season["OffRating"] * 108.35) / 108.57
team_season["DefRating"] = team_season["PA"] /( 100* (team_season["OFGA"] + team_season["OOR"] - team_season["OTO"] + team_season["OFTA"] * 0.475))
team_season["ScoreDiff"] = team_season["PF"] - team_season["PA"]
team_season["MarginPerGame"] = team_season["ScoreDiff"] / team_season["Games"]
# Rebounds and Tornover efficiency 
team_season["RebATO"] = (team_season["OR"] + team_season["TO"]) - (team_season["OOR"] + team_season["OTO"])
team_season = team_season.merge(df_seeds, on = ["Season", "TeamID"], how = "left")
team_season = team_season.merge(rank_avg,on=["Season","TeamID"],how="left")
team_season

,Season,TeamID,Games,Wins,PF,PA,FGA,OR,TO,FTA,...,AvgPA,AvgMargin,OffRating,AdjOffRating,DefRating,ScoreDiff,MarginPerGame,RebATO,SeedNum,AvgRank
0,2003,1102,28,12,1603,1596,1114,117,320,479,...,57.000000,0.250000,0.014080,0.014051,0.011822,7,0.250000,-195,NaN,154.058824
1,2003,1103,27,13,2127,2110,1508,264,341,698,...,78.148148,0.629630,0.012068,0.012043,0.012168,17,0.629630,-134,NaN,168.705882
2,2003,1104,28,17,1940,1820,1601,380,372,586,...,65.000000,4.285714,0.010279,0.010258,0.010712,120,4.285714,59,10.0,36.638889
3,2003,1105,26,7,1866,1993,1602,351,485,568,...,76.653846,-4.884615,0.010738,0.010716,0.011796,-127,-4.884615,4,NaN,308.735294
4,2003,1106,28,13,1781,1785,1548,344,477,461,...,63.750000,-0.142857,0.010900,0.010878,0.010612,-4,-0.142857,82,NaN,260.911765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14306,2026,3477,26,11,1618,1753,1530,204,315,408,...,67.423077,-5.192308,0.010032,0.010012,0.010001,-135,-5.192308,-75,NaN,NaN
14307,2026,3478,29,10,1615,2034,1500,228,409,403,...,70.137931,-14.448276,0.010692,0.010671,0.010274,-419,-14.448276,20,NaN,NaN
14308,2026,3479,28,13,1800,2074,1583,197,396,421,...,74.071429,-9.785714,0.011364,0.011341,0.011957,-274,-9.785714,-71,NaN,NaN
14309,2026,3480,27,15,1913,1892,1555,242,400,558,...,70.074074,0.777778,0.011510,0.011487,0.010229,21,0.777778,5,NaN,NaN


In [175]:

df_tourney = df_tourney_results.merge(
    team_season, left_on=["Season","WTeamID"], right_on=["Season","TeamID"], how="left"
).rename(columns={
    "Wins":"W_Wins","Losses":"W_Losses","WinPct":"W_WinPct",
    "AvgPF":"W_AvgPF","AvgPA":"W_AvgPA","AvgMargin":"W_AvgMargin",
    "OffRating":"W_OffRating","DefRating":"W_DefRating", "SeedNum":"W_SeedNum", "RebATO":"W_RebATO", "AdjOffRating":"W_AdjOffRating", "AvgRank" : "W_AvgRank"
}).drop("TeamID", axis=1)

df_tourney = df_tourney.merge(
    team_season, left_on=["Season","LTeamID"], right_on=["Season","TeamID"], how="left"
).rename(columns={
    "Wins":"L_Wins","Losses":"L_Losses","WinPct":"L_WinPct",
    "AvgPF":"L_AvgPF","AvgPA":"L_AvgPA","AvgMargin":"L_AvgMargin",
    "OffRating":"L_OffRating","DefRating":"L_DefRating", "SeedNum":"L_SeedNum","RebATO":"L_RebATO", "AdjOffRating":"L_AdjOffRating", "AvgRank": "L_AvgRank"
}).drop("TeamID", axis=1)

df_tourney["WinPct_diff"] = df_tourney["W_WinPct"] - df_tourney["L_WinPct"]
df_tourney["AvgPF_diff"] = df_tourney["W_AvgPF"] - df_tourney["L_AvgPF"]
df_tourney["AvgPA_diff"] = df_tourney["W_AvgPA"] - df_tourney["L_AvgPA"]
df_tourney["AvgRank_diff"] = df_tourney["W_AvgRank"] - df_tourney["L_AvgRank"]
df_tourney["AvgMargin_diff"] = df_tourney["W_AvgMargin"] - df_tourney["L_AvgMargin"]
df_tourney["AvgOffRating_diff"] = df_tourney["W_OffRating"] - df_tourney["L_OffRating"]
df_tourney["AvgDefRating_diff"] = df_tourney["W_DefRating"] - df_tourney["L_DefRating"]
df_tourney["AvgRebATO_diff"] = df_tourney["W_RebATO"] - df_tourney["L_RebATO"]
df_tourney["AdjOffRating_diff"] = df_tourney["W_AdjOffRating"] - df_tourney["L_AdjOffRating"]
df_tourney["Seed_diff"] = df_tourney["W_SeedNum"] - df_tourney["L_SeedNum"] if "W_SeedNum" in df_tourney.columns else 0
df_tourney["NetRating_diff"] = ((df_tourney["W_OffRating"] - df_tourney["W_DefRating"]) -(df_tourney["L_OffRating"] - df_tourney["L_DefRating"]))
df_tourney["Win"] = 1
diff_features = ["WinPct_diff", "AvgPF_diff", "AvgPA_diff","AvgOffRating_diff", "AvgDefRating_diff", "Seed_diff", "AvgRebATO_diff", "AdjOffRating_diff", "NetRating_diff", "AvgRank_diff"]

df_flipped = df_tourney.copy()
for col in diff_features:
    df_flipped[col] = -df_flipped[col]
df_flipped["Win"] = 0

df_model = pd.concat([df_tourney, df_flipped], ignore_index=True)

X = df_model[diff_features]
y = df_model["Win"]

imputer = SimpleImputer(strategy="constant", fill_value = 0)
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)
xgb_model = XGBClassifier(objective = "binary:logistic", eval_metric = "logloss", random_state = 42)
param_grid = {"n_estimators": [300, 500, 800],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}
grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
best_xgb = grid.best_estimator_

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
brier = brier_score_loss(y_test, y_prob)
print("Brier Score:", brier)
importance = pd.Series(
    best_xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
print(importance)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Accuracy: 0.6350958744915747

Confusion Matrix:
 [[349 512]
 [116 744]]

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.41      0.53       861
           1       0.59      0.87      0.70       860

    accuracy                           0.64      1721
   macro avg       0.67      0.64      0.61      1721
weighted avg       0.67      0.64      0.61      1721

Brier Score: 0.20500781829018697
Seed_diff            0.436912
AvgRank_diff         0.103408
WinPct_diff          0.089851
AvgPA_diff           0.072564
AvgPF_diff           0.071838
AvgDefRating_diff    0.047471
NetRating_diff       0.046655
AvgRebATO_diff       0.044965
AvgOffRating_diff    0.044311
AdjOffRating_diff    0.042025
dtype: float32


In [195]:
sub = pd.read_csv(data + "SampleSubmissionStage1.csv")
sub[['Season', 'Team1', 'Team2']] = sub['ID'].str.split('_', expand=True).astype(int)

In [177]:
sub = sub.merge(
    team_season, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left"
).rename(columns={
    "Wins":"W_Wins","Losses":"W_Losses","WinPct":"W_WinPct",
    "AvgPF":"W_AvgPF","AvgPA":"W_AvgPA","AvgMargin":"W_AvgMargin",
    "OffRating":"W_OffRating","DefRating":"W_DefRating",
    "SeedNum":"W_SeedNum","RebATO":"W_RebATO",
    "AdjOffRating":"W_AdjOffRating","AvgRank":"W_AvgRank"
}).drop("TeamID", axis=1)

sub = sub.merge(
    team_season, left_on=["Season","Team2"], right_on=["Season","TeamID"], how="left"
).rename(columns={
    "Wins":"L_Wins","Losses":"L_Losses","WinPct":"L_WinPct",
    "AvgPF":"L_AvgPF","AvgPA":"L_AvgPA","AvgMargin":"L_AvgMargin",
    "OffRating":"L_OffRating","DefRating":"L_DefRating",
    "SeedNum":"L_SeedNum","RebATO":"L_RebATO",
    "AdjOffRating":"L_AdjOffRating","AvgRank":"L_AvgRank"
}).drop("TeamID", axis=1)

In [179]:
diff_features = ["WinPct_diff", "AvgPF_diff", "AvgPA_diff","AvgOffRating_diff", "AvgDefRating_diff", "Seed_diff", "AvgRebATO_diff", "AdjOffRating_diff", "NetRating_diff", "AvgRank_diff"]

sub["WinPct_diff"] = sub["W_WinPct"] - sub["L_WinPct"]
sub["AvgPF_diff"] = sub["W_AvgPF"] - sub["L_AvgPF"]
sub["AvgPA_diff"] = sub["W_AvgPA"] - sub["L_AvgPA"]
sub["AvgRank_diff"] = sub["W_AvgRank"] - sub["L_AvgRank"]
sub["AvgMargin_diff"] = sub["W_AvgMargin"] - sub["L_AvgMargin"]
sub["AvgOffRating_diff"] = sub["W_OffRating"] - sub["L_OffRating"]
sub["AvgDefRating_diff"] = sub["W_DefRating"] - sub["L_DefRating"]
sub["AvgRebATO_diff"] = sub["W_RebATO"] - sub["L_RebATO"]
sub["AdjOffRating_diff"] = sub["W_AdjOffRating"] - sub["L_AdjOffRating"]
sub["Seed_diff"] = sub["W_SeedNum"] - sub["L_SeedNum"]
sub["NetRating_diff"] = (sub["W_OffRating"] - sub["W_DefRating"]) - (sub["L_OffRating"] - sub["L_DefRating"])

In [180]:
imputer = SimpleImputer(strategy="constant", fill_value=0)
X_new = imputer.fit_transform(sub[diff_features])
sub['Pred'] = best_xgb.predict_proba(X_new)[:,1]
print(sub[["ID", "Pred"]])
sub[['ID','Pred']].to_csv("submission_predictions.csv", index=False)

                    ID      Pred
0       2022_1101_1102  0.719161
1       2022_1101_1103  0.465457
2       2022_1101_1104  0.409936
3       2022_1101_1105  0.626810
4       2022_1101_1106  0.747489
...                ...       ...
519139  2025_3477_3479  0.476927
519140  2025_3477_3480  0.385441
519141  2025_3478_3479  0.416634
519142  2025_3478_3480  0.422318
519143  2025_3479_3480  0.547068

[519144 rows x 2 columns]
